# DDS-Edge: chạy lại toàn bộ theo 10 lớp (Colab)

**Thay đổi so với các lần trước:** từ giờ mô hình phân loại **10 lớp gốc c0–c9** của State Farm (không gộp thành 6 nhóm). Hệ quả cần nhớ:

| | 6 nhóm (cũ) | **10 lớp (mới)** |
|---|---|---|
| Mất cân bằng lớp (tập train) | tỉ lệ lớn/nhỏ 4,94 (`phone` 41,6%) | **1,31** (gần như cân bằng, 1.161 đến 1.520 ảnh mỗi lớp) |
| Mốc "luôn đoán lớp đông nhất" (test) | accuracy 0,387, macro-F1 0,093 | **accuracy 0,117, macro-F1 0,021** |
| Số ảnh test mỗi lớp | 225 đến 999 | **221 đến 301** |
| `class_weight` / sampler / cân bằng dữ liệu | cần thử | **không còn cần** |

**Dữ liệu không phải làm lại:** gói `sf_320x240` (bỏ ảnh trùng, resize 320×240, chia 18/4/4 tài xế) giữ nguyên. Chỉ cách đọc nhãn đổi: code tự lấy nhãn từ cột `raw_label` (c0–c9).

**Các nhãn:** c0 lái bình thường, c1 nhắn tin tay phải, c2 nghe/gọi tay phải, c3 nhắn tin tay trái, c4 nghe/gọi tay trái, c5 thao tác radio, c6 uống nước, c7 với ra sau, c8 chải tóc/trang điểm, c9 nói chuyện với hành khách.

**Kế hoạch chạy (EfficientNet-Lite0 làm trọng tâm):**

| Phần | Cell | Việc | Thời gian ước tính* |
|---|---|---|---|
| A | 1 đến 5 | Chuẩn bị môi trường | 5 phút |
| B | 6 | Xem phân bố 10 lớp và mốc đoán bừa | 1 phút |
| C | 7 | Chạy thử 1 epoch | 2 phút |
| D | 8 đến 11 | **Train 4 mô hình** (EffNet-Lite0, ResNet18, MobileNetV3-S, ResNet50) | 8 + 8 + 15 + lâu |
| E | 12 | Bảng so sánh | 1 phút |
| F | 13 đến 14 | Phân tích lỗi 10 lớp | 2 phút |
| G | 15 đến 16 | (Tuỳ chọn) cải thiện: augmentation mạnh, ảnh 320×240 | 8 đến 12 phút mỗi cái |
| H | 17 | (Tuỳ chọn) cross-validation 5 fold | khoảng 40 phút |

\*Tính từ thời gian các lần chạy trước (EfficientNet-Lite0 và ResNet18 khoảng 8 phút, MobileNetV3-S khoảng 15 phút); ResNet50 chưa đo. **Ưu tiên theo thứ tự cell**: nếu thiếu giờ thì bỏ phần G, H, rồi ResNet50.

**Cần `git push` code mới lên GitHub trước** (cell 3 chạy `git pull`); code mới đổi mặc định sang 10 lớp.

## A. Chuẩn bị môi trường

### Cell 1: kiểm tra GPU
Phải thấy tên GPU (thường là `Tesla T4`). Nếu lỗi: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Bat GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.cuda.get_device_name(0))

### Cell 2: nối Google Drive
Dùng để đọc file dữ liệu `sf_320x240.zip` và lưu kết quả train.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Cell 3: tải code từ GitHub và cài thư viện
`git clone` lần đầu, các lần sau `git pull`. Dòng cuối in commit đang dùng. Nếu chưa push thì các lệnh bên dưới sẽ train 6 nhóm (mặc định cũ): hãy push trước.

In [ ]:
%cd /content
!git clone https://github.com/ColorOfDreams/dds-edge.git 2>/dev/null || echo 'already cloned'
%cd /content/dds-edge
!git pull -q
!pip install -q timm onnx onnxruntime tabulate
!pip install -q -e . --no-deps
!git log --oneline -1

### Cell 4: giải nén dữ liệu và nối vào đúng chỗ
**Kết quả cần thấy:** `20007 images; {'test': 4, 'train': 18, 'val': 4} drivers per split`.

In [ ]:
# data: extract to the local disk (much faster than reading from Drive), then link to data/processed/sf_320x240
# accepts sf_320x240*.zip (preferred) or sf_320x240*.rar in My Drive
import os, glob, shutil, zipfile, subprocess
os.chdir('/content/dds-edge')
if not glob.glob('/content/sf_data/**/manifest.csv', recursive=True):
    arch = None
    for ext in ('zip', 'rar'):
        found = (glob.glob(f'/content/drive/MyDrive/sf_320x240*.{ext}')
                 or glob.glob(f'/content/drive/MyDrive/**/sf_320x240*.{ext}', recursive=True))
        if found:
            arch = found[0]
            break
    assert arch, 'Khong thay sf_320x240*.zip/.rar trong My Drive: da upload len Drive chua?'
    print('archive:', arch)
    shutil.rmtree('/content/sf_data', ignore_errors=True)
    os.makedirs('/content/sf_data')
    if arch.endswith('.zip'):
        with zipfile.ZipFile(arch) as z:
            for info in z.infolist():
                name = info.filename.replace(chr(92), '/')  # zips made on Windows use backslashes
                if name.endswith('/'):
                    continue
                dst = os.path.join('/content/sf_data', name)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(info) as f, open(dst, 'wb') as g:
                    shutil.copyfileobj(f, g)
    else:  # .rar needs the unrar tool
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'unrar'], check=True)
        subprocess.run(['unrar', 'x', '-idq', '-y', arch, '/content/sf_data/'], check=True)
hits = glob.glob('/content/sf_data/**/manifest.csv', recursive=True)
assert hits, 'Khong thay manifest.csv sau khi giai nen: ' + str(os.listdir('/content/sf_data')[:5])
src = os.path.dirname(hits[0])
os.makedirs('data/processed', exist_ok=True)
link = 'data/processed/sf_320x240'
if os.path.islink(link):  # stale link from an earlier run
    os.unlink(link)
assert not os.path.exists(link), f'{link} is a real folder, remove it first'
os.symlink(src, link)
import pandas as pd
m = pd.read_csv(link + '/manifest.csv')
print(len(m), 'images;', m.groupby('split')['group'].nunique().to_dict(), 'drivers per split')  # expect 20007; test 4, train 18, val 4

### Cell 5: lưu kết quả train vào Drive
`runs/` được nối vào `MyDrive/dds-edge-runs/`. Các run mới có đuôi `_f10` (10 lớp) nên không lẫn với các run cũ (6 nhóm).

In [ ]:
# write runs/ straight to Drive so results survive a disconnect
%cd /content/dds-edge
!mkdir -p /content/drive/MyDrive/dds-edge-runs
!rm -rf runs
!ln -s /content/drive/MyDrive/dds-edge-runs runs
!ls -la runs | head -3

## B. Nhìn vào dữ liệu 10 lớp

### Cell 6: số ảnh mỗi lớp theo tập, và mốc đoán bừa
In bảng 10 lớp × 3 tập (train / val / test) và **mốc "luôn đoán lớp đông nhất ở tập train"**. Mốc này là điểm của mô hình ngốc nhất: mọi mô hình phải vượt xa nó. Với 10 lớp, mốc rất thấp (macro-F1 khoảng 0,02) vì đoán bừa chỉ đúng khoảng 1/10 ảnh.

**Cần để ý:** các lớp gần bằng nhau (tỉ lệ lớn/nhỏ 1,31), nên accuracy ở đây ít bị đánh lừa hơn so với bản 6 nhóm; macro-F1 vẫn là điểm chính.

In [ ]:
import pandas as pd
m = pd.read_csv('/content/dds-edge/data/processed/sf_320x240/manifest.csv')
t = m.pivot_table(index='raw_label', columns='split', values='path', aggfunc='count')[['train', 'val', 'test']]
t.loc['tổng'] = t.sum()
print(t.to_string())
tr = m[m.split == 'train'].raw_label.value_counts()
print(f'\ntỉ lệ lớn/nhỏ ở train: {tr.max() / tr.min():.2f}  (lớp đông nhất: {tr.idxmax()})')
for s in ('val', 'test'):
    p = (m[m.split == s].raw_label == tr.idxmax()).mean()
    print(f"luôn đoán {tr.idxmax()} trên {s}: accuracy {p:.3f}, macro-F1 {(2 * p / (1 + p)) / 10:.3f}")

## C. Chạy thử

### Cell 7: train thử 1 epoch (`smoke10`)
Chỉ để kiểm tra đường ống chạy thông với 10 lớp. **Không dùng kết quả này để báo cáo.** Dòng JSON cuối phải có `"n_params": 3383818` (EfficientNet-Lite0 với 10 đầu ra) và `test_f1_per_class` có **10 khoá c0 đến c9** (nếu chỉ thấy 6 nhóm tên tiếng Anh nghĩa là code mới chưa được `git pull`).

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch efficientnet_lite0 --epochs 1 --name smoke10

## D. Huấn luyện bốn mô hình (10 epoch, giao thức giống mọi lần trước)

Giao thức cố định để so sánh công bằng: pretrained ImageNet, AdamW lr 0,001, weight decay 1e-4, cosine, batch 64, label smoothing 0,1, augmentation cơ bản, seed 42, chọn epoch theo **val macro-F1 của 10 lớp**, test chấm một lần. Mỗi cell dán **thẳng lệnh** (không dùng biến) để không nhầm tên run. Sau mỗi lần xong, kiểm tra `n_params` ở dòng JSON cuối (số tham số hơi lớn hơn bản 6 nhóm vì lớp cuối có 10 đầu ra thay vì 6).

### Cell 8: `effl0_f10`: EfficientNet-Lite0 (ứng viên edge chính, ưu tiên số 1)
Khoảng 8 phút. `n_params` phải là **3383818**.

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch efficientnet_lite0 --name effl0_f10

### Cell 9: `resnet18_f10`: ResNet18 (mốc nặng vừa)
Khoảng 8 phút. `n_params` phải là **11181642**.

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch resnet18 --name resnet18_f10

### Cell 10: `mnv3_f10`: MobileNetV3-Small (mô hình nhỏ nhất)
Khoảng 15 phút. `n_params` phải là **1528106**. Run này quan trọng cho phần INT8: lần trước MobileNetV3-Small sụp sau lượng tử hoá, cần xem 10 lớp có còn vậy không.

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch mobilenet_v3_small --name mnv3_f10

### Cell 11: `resnet50_f10`: ResNet50 (mốc nặng, làm cuối)
Nặng nhất nên lâu nhất (chưa đo, có thể 15 đến 25 phút). `n_params` phải là **23528522**. **Bỏ qua nếu thiếu giờ**; khi đó ResNet50 chỉ có số tham số và MACs, không có điểm trong bài 10 lớp.

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch resnet50 --name resnet50_f10

## E. So sánh

### Cell 12: bảng so sánh các mô hình
Đọc `metrics.json` của bốn run `_f10` và in: tham số, val macro-F1, test accuracy, **test macro-F1** (điểm chính), **lớp yếu nhất** và F1 của nó, và **khoảng cách val − test** (càng lớn càng đáng ngờ). Run nào chưa chạy ghi `THIẾU`.

**Nhớ:** chênh dưới khoảng 0,03 giữa hai mô hình chưa chắc là thật (nhiễu giữa các lần chạy).

In [ ]:
import json, os
RUNS = '/content/dds-edge/runs'
names = ['effl0_f10', 'resnet18_f10', 'mnv3_f10', 'resnet50_f10']
print(f"{'run':14s}{'tham số':>11s}{'val F1':>8s}{'test acc':>10s}{'test F1':>9s}  {'lớp yếu nhất':14s}{'val-test':>9s}")
for n in names:
    p = f'{RUNS}/{n}/metrics.json'
    if not os.path.exists(p):
        print(f'{n:14s}  THIẾU'); continue
    m = json.load(open(p)); pc = m['test_f1_per_class']; w = min(pc, key=pc.get)
    print(f"{n:14s}{m['n_params']:11d}{m['best_val_macro_f1']:8.3f}{m['test_acc']:10.3f}{m['test_macro_f1']:9.3f}  {w + f' {pc[w]:.2f}':14s}{m['best_val_macro_f1'] - m['test_macro_f1']:9.3f}")

## F. Phân tích lỗi (10 lớp)

### Cell 13: confusion matrix và F1 từng lớp của `effl0_f10`
In ma trận nhầm lẫn 10×10 (hàng = nhãn thật, cột = nhãn đoán), F1 từng lớp, **các nhầm lẫn nhiều nhất** và độ chính xác từng tài xế test. Câu hỏi chính: với 10 lớp, **cặp nào hay bị nhầm** (ví dụ c1 và c3 là nhắn tin tay phải và tay trái; c2 và c4 là nghe gọi tay phải và trái; c9 với c7/c8)? Đây là chỗ bản 6 nhóm từng che mất.

In [ ]:
%cd /content/dds-edge
!python -m dds_edge.dl.analyze --run effl0_f10

### Cell 14: xem ảnh của cặp bị nhầm nhiều nhất
Tự tìm cặp (nhãn thật, nhãn đoán) bị nhầm nhiều nhất của `effl0_f10`, rồi lưu và hiện một ảnh ghép 12 ảnh. Bạn nhìn bằng mắt xem hai lớp đó thật sự khác nhau không, hay ảnh nhập nhằng.

In [ ]:
import pandas as pd
from IPython.display import Image, display
pr = pd.read_csv('/content/dds-edge/runs/effl0_f10/predictions.csv')
bad = pr[pr.true != pr.pred].groupby(['true', 'pred']).size().sort_values(ascending=False)
print('Top 5 nhầm lẫn (thật -> đoán, số ảnh):'); print(bad.head(5).to_string())
T, P = bad.index[0]
print(f'\nẢnh ghép: {T} bị đoán là {P}')
!python -m dds_edge.dl.analyze --run effl0_f10 --montage {T} {P} --n 12 > /dev/null
display(Image(f'/content/dds-edge/runs/effl0_f10/mis_{T}_as_{P}.jpg'))

## G. (Tuỳ chọn) Thử cải thiện, mỗi lần đổi một thứ

Chỉ làm nếu còn giờ. Bỏ `#` ở **đúng một dòng** rồi chạy cell. So với `effl0_f10`; chênh dưới khoảng 0,03 chưa kết luận.

### Cell 15: augmentation mạnh hơn (`effl0_f10_aug2`)
Cắt rộng hơn, xoay 12°, đổi màu mạnh, làm mờ, xoá ngẫu nhiên một mảng ảnh. Mục đích: bớt học thuộc đặc điểm từng tài xế.

In [ ]:
%cd /content/dds-edge
# !python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch efficientnet_lite0 --aug_level 2 --name effl0_f10_aug2

### Cell 16: ảnh 320×240 giữ tỉ lệ 4:3 (`effl0_f10_sz320x240`)
Đưa ảnh vào mạng ở độ phân giải gốc, không bị ép vuông. Tốn tính toán hơn khoảng 1,6 lần, nên chậm hơn lúc train và lúc suy luận.

In [ ]:
%cd /content/dds-edge
# !python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch efficientnet_lite0 --img_size 320x240 --name effl0_f10_sz320x240

## H. (Tuỳ chọn) Cross-validation theo tài xế

### Cell 17: 5 fold cho EfficientNet-Lite0
Chia 26 tài xế thành 5 nhóm; mỗi fold 5 đến 6 tài xế làm test, 4 làm val, còn lại train; **mỗi tài xế làm test đúng một lần**. Cho điểm ổn định hơn nhiều so với một lần chia (chỉ 4 tài xế test). Khoảng 40 phút. Sau khi xong, tải 5 thư mục `effl0_f10_cv0` đến `effl0_f10_cv4` về máy rồi chạy `python -m dds_edge.dl.cv_summary --prefix effl0_f10_cv --runs_dir <thư mục>` để có trung bình và độ lệch chuẩn.

Bỏ `#` ở các dòng muốn chạy (cả 5 nếu đủ giờ). Mỗi dòng cần **cả hai cờ** `--cv_folds 5` và `--cv_fold i`.

In [ ]:
%cd /content/dds-edge
# !python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch efficientnet_lite0 --cv_folds 5 --cv_fold 0 --name effl0_f10_cv0
# !python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch efficientnet_lite0 --cv_folds 5 --cv_fold 1 --name effl0_f10_cv1
# !python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch efficientnet_lite0 --cv_folds 5 --cv_fold 2 --name effl0_f10_cv2
# !python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch efficientnet_lite0 --cv_folds 5 --cv_fold 3 --name effl0_f10_cv3
# !python -u -m dds_edge.dl.train --grouping full10 --manifest processed/sf_320x240/manifest.csv --workers 2 --arch efficientnet_lite0 --cv_folds 5 --cv_fold 4 --name effl0_f10_cv4

## Sau khi chạy xong: việc làm trên máy bạn
1. Từ Drive `dds-edge-runs/`, tải các thư mục run `_f10` về `D:\work\dds-edge\drive\dds-edge-runs\` (cần ít nhất `best.pt`, `metrics.json`, `predictions.csv`). Đợi Drive đồng bộ xong trước khi tải.
2. Lượng tử hoá INT8 và đo tốc độ cho từng mô hình (PowerShell, trong `D:\work\dds-edge`):
   `python -m dds_edge.dl.export_quantize --run effl0_f10 --runs_dir drive/dds-edge-runs --manifest processed/sf_320x240/manifest.csv --threads 1`
   (làm tương tự cho `resnet18_f10`, `mnv3_f10`; thêm `--calib_method percentile --tag percentile` nếu INT8 tụt điểm).
3. Bảng và biểu đồ tổng hợp: `python -m dds_edge.dl.summarize --runs_dir drive/dds-edge-runs`.
4. Báo kết quả để cập nhật runlog, slide và bài báo sang 10 lớp (các số 6 nhóm cũ trong slide và bài báo sẽ phải thay).